# PV056 — Plant Disease Classification with Triplet Loss

**Course**: PV056 Machine Learning and Data Mining, MUNI 2026  
**Task**: Classify plant diseases (subtask a) and detect unknown diseases (subtask b) using metric learning on PlantVillage.  
**Repo**: https://github.com/MrJoeKr/pv056-project-2026

> **Runtime**: Set to **GPU** (Runtime → Change runtime type → T4 GPU) before running.  
> **Fast mode** (default): ResNet18 + reduced config, full notebook runs in ~10 min on a T4. Set `FAST_MODE = False` in section 4 to use the full ResNet50 config from the report.

---
## Contents
1. [Setup](#setup) — clone repo, install dependencies
2. [Dataset](#dataset) — mount Drive, point to PlantVillage
3. [EDA](#eda) — class distribution, outlier detection
4. [Training](#training) — stratified CV with triplet loss (Fast mode toggle)
5. [Evaluation](#evaluation) — confusion matrix, Grad-CAM, UMAP, per-class F1
6. [Unknown Detection](#unknown) — Mahalanobis distance, ROC, UMAP

---
## 1. Setup <a id='setup'></a>

In [ ]:
# Clone the project repository
!git clone https://github.com/YOUR_USERNAME/pv056-project-2026.git
%cd pv056-project-2026

In [ ]:
# Install PyTorch with CUDA and remaining dependencies
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
!pip install -r requirements.txt -q

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

---
## 2. Dataset <a id='dataset'></a>

The PlantVillage dataset must be available at `data/PlantVillage/`.  
Mount your Google Drive and symlink the dataset folder, or upload it directly.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Adjust this path to wherever PlantVillage lives on your Drive
DRIVE_DATASET_PATH = '/content/drive/MyDrive/PlantVillage'

os.makedirs('data', exist_ok=True)
if not os.path.exists('data/PlantVillage'):
    os.symlink(DRIVE_DATASET_PATH, 'data/PlantVillage')

classes = os.listdir('data/PlantVillage')
print(f'Dataset ready — {len(classes)} classes found')

---
## 3. Exploratory Data Analysis <a id='eda'></a>

- **R1a**: Class label distribution
- **R1b**: Pixel-level outlier detection (z-score)

In [ ]:
!python scripts/01_eda.py

In [ ]:
from IPython.display import display, Image
import glob

for path in sorted(glob.glob('results/eda/*.png')):
    print(path)
    display(Image(path))

---
## 4. Training — Stratified CV <a id='training'></a>

- **R2a**: HPO was run separately (Optuna, 30 trials); best params baked into `src/config.py`
- **R2b**: Training with early stopping; plots training curves per fold

### Fast mode vs full-quality

The full run (ResNet50, 224×224, 50 epochs, 5 folds) takes ~60–90 min on a T4 — too slow for a Colab demo. The cell below writes a **`config_override.json`** that every script in this project honors, swapping in a fast configuration (ResNet18, 128×128, 10 epochs, 2 folds, mixed precision). Set `FAST_MODE = False` to run the full config used in the report.

> The authoritative numbers in the report were produced locally with ResNet50. This notebook reproduces the full pipeline end-to-end as a runnable demo; see the [GitHub repo](https://github.com/MrJoeKr/pv056-project-2026) for raw results tables and full-quality checkpoints.

In [ ]:
import json, os

FAST_MODE = True  # set to False to use Config() defaults (ResNet50, 224x224, 50 epochs, 5 folds)

override_path = 'results/tables/config_override.json'
os.makedirs(os.path.dirname(override_path), exist_ok=True)

if FAST_MODE:
    overrides = {
        'backbone': 'resnet18',
        'img_size': 128,
        'epochs': 10,
        'patience': 3,
        'n_folds': 2,
        'batch_size': 128,
        'use_amp': True,
    }
    with open(override_path, 'w') as f:
        json.dump(overrides, f, indent=2)
    print('Fast mode active — overrides written:')
    print(json.dumps(overrides, indent=2))
else:
    if os.path.exists(override_path):
        os.remove(override_path)
    print('Full mode — using Config() defaults')

In [ ]:
!python scripts/02_train.py

In [ ]:
for path in sorted(glob.glob('results/training_curves/*.png')):
    print(path)
    display(Image(path))

---
## 5. Evaluation <a id='evaluation'></a>

- **R3a**: Confusion matrix, per-class F1, Grad-CAM explainability, UMAP embedding visualization
- **R3b**: Summary table, statistical context

In [ ]:
!python scripts/04_evaluate.py

In [ ]:
import pandas as pd

# Summary table
summary = pd.read_csv('results/results_summary.csv')
display(summary)

# Plots
for path in ['results/confusion_matrix.png',
             'results/per_class_f1.png',
             'results/umap_embeddings.png',
             'results/gradcam_samples.png']:
    if os.path.exists(path):
        print(path)
        display(Image(path))

---
## 6. Unknown Disease Detection <a id='unknown'></a>

Subtask b: `Tomato_Bacterial_spot` is excluded from training and treated as the unknown class.  
Detection uses **Mahalanobis distance** from test embeddings to class prototypes (not softmax).

Results from our run:
- AUROC: **0.9795**, PR-AUC: **0.9627**
- Mann-Whitney U p ≈ 0.00 (unknown distances stochastically larger, highly significant)

In [ ]:
!python scripts/05_unknown.py

In [ ]:
for path in sorted(glob.glob('results/unknown_*.png')):
    print(path)
    display(Image(path))